# AttentionBench — Kaggle Benchmark Evaluation

**Can models both filter distractions AND maintain focus over extended tasks?**

This benchmark tests two complementary attention faculties:
1. **Signal-in-Noise (Selective Attention)** — Can the model find answers buried in increasing noise?
2. **Vigilance Decrement (Sustained Attention)** — Does accuracy decay over 100 repeated sub-tasks?

Task difficulty is held constant — any degradation is purely attentional.

In [ ]:
# --------------------------------------------------------------------------------
# SETUP: Generate dataset and load
# Quick Start: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/quick_start.md
# --------------------------------------------------------------------------------

import subprocess, os, sys, json, re
import pandas as pd
import numpy as np
import kaggle_benchmarks as kbench
from urllib.request import urlretrieve

# Download and run generator from GitHub (no git required)
REPO_RAW = "https://raw.githubusercontent.com/42euge/attention-bench/main"

os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)

if not os.path.exists("src/generate.py"):
    print("Downloading generator...")
    urlretrieve(f"{REPO_RAW}/src/generate.py", "src/generate.py")

if not os.path.exists("data/signal_in_noise.json"):
    print("Generating dataset...")
    subprocess.run([sys.executable, "src/generate.py"], check=True)

with open("data/signal_in_noise.json") as f:
    sin_data = json.load(f)
with open("data/vigilance.json") as f:
    vig_data = json.load(f)

sin_df = pd.DataFrame(sin_data)
vig_df = pd.DataFrame(vig_data)
print(f"Loaded {len(sin_df)} SIN + {len(vig_df)} vigilance items")

In [ ]:
# --------------------------------------------------------------------------------
# HELPERS: Answer parsing and flexible matching
# --------------------------------------------------------------------------------

def strip_thinking(response: str) -> str:
    """Strip <think>...</think> chain-of-thought blocks from reasoning models."""
    if "</think>" in response:
        return response.split("</think>")[-1].strip()
    return response

def parse_numbered_answers(response: str, count: int) -> list[str]:
    """Parse numbered answers (1. xxx, 2. xxx) from model response.
    
    Handles reasoning models that add preamble lines like
    'Here are the answers:' before the numbered list.
    """
    response = strip_thinking(response)
    lines = response.strip().split("\n")
    answers = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Only accept lines that start with a number prefix (1. 2) 3: etc.)
        # This skips preamble like "Here are the answers:" or "Based on the text:"
        if re.match(r"^\d+[\.\)\:\-]", line):
            cleaned = re.sub(r"^\d+[\.\)\:\-]\s*", "", line)
            if cleaned:
                answers.append(cleaned)
    # Fallback: if no numbered lines found, try all non-empty lines
    if not answers:
        for line in lines:
            line = line.strip()
            if line:
                answers.append(line)
    while len(answers) < count:
        answers.append("")
    return answers[:count]

def check_answer(expected: str, actual: str) -> bool:
    """Flexible answer matching: case-insensitive, substring, number formats."""
    exp = expected.lower().strip()
    act = actual.lower().strip()
    if exp == act or exp in act:
        return True
    exp_core = re.sub(r"^(approximately|about|roughly|around)\s+", "", exp)
    if exp_core in act:
        return True
    exp_d = re.sub(r"[,\s]", "", exp)
    act_d = re.sub(r"[,\s]", "", act)
    return exp_d == act_d and exp_d != ""

## Task Definitions

Two tasks, each returning `tuple[int, int]` (correct, total) for the leaderboard.

In [ ]:
# --------------------------------------------------------------------------------
# STEP 1: DEFINE TASKS
# Each task takes `llm` as first parameter. Returns (correct, total).
# Kaggle automatically swaps in different models via the "Add Models" button.
# --------------------------------------------------------------------------------

@kbench.task(name="attention_selective",
             description="Signal-in-Noise: find answers in a passage buried in noise")
def attention_selective(llm, prompt: str, answers: str, num_questions: int) -> tuple[int, int]:
    response = llm.prompt(prompt)
    expected = json.loads(answers) if isinstance(answers, str) else answers
    num_q = int(num_questions)
    parsed = parse_numbered_answers(response, num_q)
    correct = sum(1 for e, a in zip(expected, parsed) if check_answer(e, a))
    return (correct, num_q)

@kbench.task(name="attention_sustained",
             description="Vigilance Decrement: does accuracy decay over 100 repeated sub-tasks?")
def attention_sustained(llm, prompt: str, answers: str, num_subtasks: int) -> tuple[int, int]:
    response = llm.prompt(prompt)
    expected = json.loads(answers) if isinstance(answers, str) else answers
    num_sub = int(num_subtasks)
    parsed = parse_numbered_answers(response, num_sub)
    correct = sum(1 for e, a in zip(expected, parsed) if check_answer(e, a))
    return (correct, num_sub)

In [ ]:
# --------------------------------------------------------------------------------
# STEP 2: PREPARE EVALUATION DATA
# --------------------------------------------------------------------------------

MAX_ITEMS_PER_CONFIG = 2   # per (noise_type, noise_ratio). 2 × 18 configs = 36 items
SKIP_HIGH_RATIOS = False   # Include 50:1 and 100:1 — that's where discrimination lives

# SIN subset
sin_eval = sin_df.copy()
if SKIP_HIGH_RATIOS:
    sin_eval = sin_eval[sin_eval["noise_ratio"] <= 25]
sin_eval = sin_eval.groupby(["noise_type", "noise_ratio"]).head(MAX_ITEMS_PER_CONFIG).reset_index(drop=True)
sin_eval["answers"] = sin_eval["answers"].apply(json.dumps)
sin_eval_df = sin_eval[["prompt", "answers", "num_questions"]].copy()

# Vigilance data (all 6 items)
vig_eval = vig_df.copy()
vig_eval["answers"] = vig_eval["answers"].apply(json.dumps)
vig_eval_df = vig_eval[["prompt", "answers", "num_subtasks"]].copy()

print(f"SIN: {len(sin_eval_df)} items | Vigilance: {len(vig_eval_df)} items")

## Run Evaluation

Uses `kbench.llm` — the default model. Use Kaggle's **"Add Models"** button to run across multiple models.

In [ ]:
# --------------------------------------------------------------------------------
# STEP 3: RUN SIGNAL-IN-NOISE EVALUATION
# Uses default model. Kaggle swaps models via "Add Models" UI.
# --------------------------------------------------------------------------------

print(f"Running {len(sin_eval_df)} SIN items...")
sin_runs = attention_selective.evaluate(
    llm=kbench.llm,
    evaluation_data=sin_eval_df,
    n_jobs=2,
)
sin_results_df = sin_runs.as_dataframe()

# Convert (correct, total) tuples to accuracy ratio
sin_results_df["accuracy"] = sin_results_df["result"].apply(
    lambda r: r[0] / r[1] if isinstance(r, tuple) and r[1] > 0 else float(r) if not isinstance(r, tuple) else 0
)

# Merge back metadata for analysis
sin_results_df = sin_results_df.merge(
    sin_eval[["prompt", "noise_type", "noise_ratio", "passage_id", "domain", "task_id"]],
    on="prompt", how="left"
)
print(f"Collected {len(sin_results_df)} SIN results")

In [ ]:
# --------------------------------------------------------------------------------
# STEP 4: RUN VIGILANCE EVALUATION
# Uses default model. Kaggle swaps models via "Add Models" UI.
# --------------------------------------------------------------------------------

print(f"Running {len(vig_eval_df)} vigilance items...")
vig_runs = attention_sustained.evaluate(
    llm=kbench.llm,
    evaluation_data=vig_eval_df,
    n_jobs=2,
)
vig_results_df = vig_runs.as_dataframe()

# Convert (correct, total) tuples to accuracy ratio
vig_results_df["accuracy"] = vig_results_df["result"].apply(
    lambda r: r[0] / r[1] if isinstance(r, tuple) and r[1] > 0 else float(r) if not isinstance(r, tuple) else 0
)

# Merge back metadata
vig_results_df = vig_results_df.merge(
    vig_eval[["prompt", "task_type", "variant", "task_id", "oddball_position"]],
    on="prompt", how="left"
)
print(f"Collected {len(vig_results_df)} vigilance results")

## Results & Analysis

In [ ]:
# Summary statistics

print("=== Signal-in-Noise Results ===")
print(sin_results_df.groupby(["noise_type", "noise_ratio"])["accuracy"].mean().unstack().to_string())

print("\n=== Attention Thresholds (highest ratio with ≥80% accuracy) ===")
for ntype in ["unrelated", "related", "adversarial"]:
    subset = sin_results_df[sin_results_df["noise_type"] == ntype]
    curve = subset.groupby("noise_ratio")["accuracy"].mean()
    threshold = 0
    for ratio in sorted(curve.index):
        if curve[ratio] >= 0.8:
            threshold = ratio
    print(f"  {ntype:12s} | threshold = {threshold}:1")

print("\n=== Vigilance Results ===")
print(vig_results_df.groupby(["task_type", "variant"])["accuracy"].mean().to_string())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

# SIN: Accuracy curves by noise ratio
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, ntype in zip(axes, ["unrelated", "related", "adversarial"]):
    subset = sin_results_df[sin_results_df["noise_type"] == ntype]
    curve = subset.groupby("noise_ratio")["accuracy"].mean()
    ax.plot(curve.index, curve.values, "o-", linewidth=2)
    ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.5, label="80% threshold")
    ax.set_xscale("log")
    ax.set_xlabel("Noise ratio")
    ax.set_title(f"{ntype}")
    ax.set_xticks(sorted(subset["noise_ratio"].unique()))
    ax.set_xticklabels(sorted(subset["noise_ratio"].unique()))
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8, loc="lower left")
axes[0].set_ylabel("Accuracy")
fig.suptitle("Signal-in-Noise: Accuracy vs Noise Ratio", fontsize=14)
plt.tight_layout()
plt.savefig("sin_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to sin_results.png")

## Next Steps

1. Click **"Save Task"** (top right) to publish to the leaderboard.
2. Use **"Add Models"** to run across multiple frontier models.
3. Compare attention thresholds across models to measure discriminatory power.